# Remaining Useful Life Prediction of Turbofan Engines
## Metaheuristic-Optimised LSTM with SHAP-Based Explainability

**Paper:** *Remaining Useful Life Prediction of Turbofan Engines Using Metaheuristic-Optimised LSTM with SHAP-Based Explainability*  
**Authors:** Sultan Zeybek, Büşra Öztürk  
**Dataset:** NASA C-MAPSS (FD001–FD004)  

This notebook reproduces all experiments reported in the paper:
- Data loading and preprocessing (FD001–FD004)
- Baseline LSTM training
- PSO-based LSTM hyperparameter optimisation
- GA-based LSTM hyperparameter optimisation
- BiLSTM fixed-configuration comparator
- Multi-run statistical validation (N=10 / N=5)
- SHAP explainability analysis
- Convergence and statistical figures


## 1. Dependencies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, time, random, warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler
import shap

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## 2. Configuration

Set `DATA_DIR` to the folder containing the C-MAPSS `.txt` files.  
The dataset is publicly available at: https://data.nasa.gov/Aerospace/CMAPSS-Jet-Engine-Simulated-Data/ff5v-kuh6


In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR  = './data/CMAPSS'   # <-- update to your local path
OUT_DIR   = './outputs'
os.makedirs(OUT_DIR, exist_ok=True)

DATASETS  = ['FD001', 'FD002', 'FD003', 'FD004']

# ── Training hyperparameters ────────────────────────────────────────────────
WINDOW    = 30       # sliding window length
MAX_RUL   = 125      # piecewise-linear RUL cap
N_RUNS    = 10       # independent runs for FD001/FD003
N_RUNS_MULTI = 5     # independent runs for FD002/FD004 (higher cost)
EPOCHS    = 50
PATIENCE  = 10
OPT_EPOCHS = 10     # epochs during optimisation fitness evaluation

# ── Search space ────────────────────────────────────────────────────────────
NEURON_CHOICES = [16, 32, 64, 128]
BATCH_CHOICES  = [32, 64, 128]
LR_BOUNDS      = (0.0001, 0.01)

# ── Optimisation budget ─────────────────────────────────────────────────────
POP_SIZE  = 5
MAX_ITER  = 20    # PSO iterations / GA generations

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 3. Data Loading and Preprocessing

In [ ]:
COLUMN_NAMES = (
    ['engine_id', 'cycle'] +
    [f'op{i}' for i in range(1, 4)] +
    [f's{i}' for i in range(1, 22)]
)

# Features excluded after Pearson correlation analysis (near-zero variance)
DROP_SENSORS = ['s1', 's5', 's6', 's10', 's16', 's17', 's18', 's19']

def load_cmapss(dataset, data_dir=DATA_DIR):
    """Load raw C-MAPSS train/test/RUL files."""
    train = pd.read_csv(f'{data_dir}/train_{dataset}.txt',
                        sep='\s+', header=None, names=COLUMN_NAMES)
    test  = pd.read_csv(f'{data_dir}/test_{dataset}.txt',
                        sep='\s+', header=None, names=COLUMN_NAMES)
    rul   = pd.read_csv(f'{data_dir}/RUL_{dataset}.txt',
                        sep='\s+', header=None, names=['RUL'])
    return train, test, rul

def preprocess(dataset, window=WINDOW, max_rul=MAX_RUL):
    """Full preprocessing pipeline: RUL labelling, feature selection,
    normalisation, and sliding-window sequence construction."""
    train, test, rul_df = load_cmapss(dataset)

    # Piecewise-linear RUL labelling
    max_cycles = train.groupby('engine_id')['cycle'].max().reset_index()
    max_cycles.columns = ['engine_id', 'max_cycle']
    train = train.merge(max_cycles, on='engine_id')
    train['RUL'] = (train['max_cycle'] - train['cycle']).clip(upper=max_rul)
    train.drop(columns=['max_cycle'], inplace=True)

    # Feature selection
    drop_cols = DROP_SENSORS.copy()
    if dataset in ['FD001', 'FD003']:       # constant op settings
        drop_cols += ['op1', 'op2', 'op3']
    feature_cols = [c for c in COLUMN_NAMES
                    if c not in ['engine_id', 'cycle'] + drop_cols]

    # Min-max normalisation (fit on train, apply to test)
    scaler = MinMaxScaler()
    train[feature_cols] = scaler.fit_transform(train[feature_cols])
    test[feature_cols]  = scaler.transform(test[feature_cols])

    # Sliding-window sequences — training
    X_train, y_train = [], []
    for eng_id, grp in train.groupby('engine_id'):
        vals = grp[feature_cols].values
        ruls = grp['RUL'].values
        for i in range(len(vals) - window + 1):
            X_train.append(vals[i:i+window])
            y_train.append(ruls[i+window-1])

    # Last-window sequences — testing (with zero-padding for short sequences)
    X_test, y_test = [], []
    for eng_id, grp in test.groupby('engine_id'):
        vals = grp[feature_cols].values
        if len(vals) >= window:
            X_test.append(vals[-window:])
        else:
            pad = np.zeros((window - len(vals), len(feature_cols)))
            X_test.append(np.vstack([pad, vals]))
    y_test = rul_df['RUL'].values

    return (np.array(X_train), np.array(y_train),
            np.array(X_test),  np.array(y_test))

## 4. Metrics and Model Builder

In [ ]:
def compute_metrics(y_true, y_pred):
    """Return RMSE, MAE, R² as a dict."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2}


def build_lstm(neurons, lr, input_shape=None, bidirectional=False):
    """Build a 2-layer LSTM (or BiLSTM) regression model.

    Args:
        neurons: number of units per LSTM layer.
        lr: Adam learning rate.
        input_shape: (timesteps, features) — inferred from global X_train if None.
        bidirectional: if True, wrap both LSTM layers with Bidirectional.
    Returns:
        Compiled Keras model.
    """
    if input_shape is None:
        input_shape = (X_train.shape[1], X_train.shape[2])
    model = Sequential()
    layer1 = LSTM(neurons, return_sequences=True, input_shape=input_shape)
    layer2 = LSTM(neurons)
    if bidirectional:
        model.add(Bidirectional(layer1))
        model.add(Dropout(0.2))
        model.add(Bidirectional(layer2))
    else:
        model.add(layer1)
        model.add(Dropout(0.2))
        model.add(layer2)
    model.add(Dropout(0.2))
    model.add(Dense(1, activation='linear'))   # linear output for regression
    model.compile(optimizer=Adam(lr), loss='mse')
    return model


def train_model(model, X_tr, y_tr, batch_size=32,
                epochs=EPOCHS, patience=PATIENCE):
    """Fit model with early stopping; return training history."""
    cb = EarlyStopping(monitor='val_loss', patience=patience,
                       restore_best_weights=True)
    hist = model.fit(X_tr, y_tr,
                     validation_split=0.1,
                     epochs=epochs,
                     batch_size=batch_size,
                     callbacks=[cb],
                     verbose=0)
    return hist

## 5. PSO-Based Hyperparameter Optimisation

Each PSO particle is a continuous vector `[lr, neurons_continuous, batch_continuous]`.  
Discrete values are obtained by nearest-neighbour mapping to `NEURON_CHOICES` / `BATCH_CHOICES`  
(see Section 3.2 of the paper for the full encoding description).


In [ ]:
class PSO:
    """
    Particle Swarm Optimisation for LSTM hyperparameter search.

    Search space (continuous):
        lr      : [0.0001, 0.01]
        neurons : continuous proxy for NEURON_CHOICES = {16, 32, 64, 128}
        batch   : continuous proxy for BATCH_CHOICES  = {32, 64, 128}

    Nearest-neighbour decoding maps continuous values to discrete sets.
    Boundary violations are corrected with a diversity-preserving strategy
    (shift bound by Delta_min = 16, reverse velocity by 0.5).
    """

    def __init__(self, swarm_size=POP_SIZE, max_iter=MAX_ITER,
                 omega=0.6, c1=1.0, c2=1.0):
        self.swarm_size = swarm_size
        self.max_iter   = max_iter
        self.omega = omega
        self.c1    = c1
        self.c2    = c2
        self.convergence = []   # best fitness per iteration

    # ── encoding helpers ──────────────────────────────────────────────────────
    @staticmethod
    def _nearest(val, choices):
        return min(choices, key=lambda x: abs(x - val))

    def _decode(self, particle):
        lr      = np.clip(particle[0], LR_BOUNDS[0], LR_BOUNDS[1])
        neurons = self._nearest(particle[1], NEURON_CHOICES)
        batch   = self._nearest(particle[2], BATCH_CHOICES)
        return lr, neurons, batch

    # ── fitness ───────────────────────────────────────────────────────────────
    def _fitness(self, particle):
        lr, neurons, batch = self._decode(particle)
        model = build_lstm(neurons, lr)
        train_model(model, X_train, y_train,
                    batch_size=batch, epochs=OPT_EPOCHS, patience=5)
        y_pred = model.predict(X_test, verbose=0).flatten()
        return np.sqrt(mean_squared_error(y_test, y_pred))

    # ── boundary correction ───────────────────────────────────────────────────
    @staticmethod
    def _fix_bounds(pos, vel, lb, ub, delta_min=16):
        for i in range(len(pos)):
            if pos[i] > ub[i]:
                ub[i] = max(ub[i] - delta_min, lb[i] + delta_min)
                pos[i] = ub[i]
                vel[i] *= -0.5
            elif pos[i] < lb[i]:
                lb[i] = min(lb[i] + delta_min, ub[i] - delta_min)
                pos[i] = lb[i]
                vel[i] *= -0.5
        return pos, vel, lb, ub

    # ── main loop ─────────────────────────────────────────────────────────────
    def run(self, verbose=False):
        lb = np.array([LR_BOUNDS[0], min(NEURON_CHOICES), min(BATCH_CHOICES)], float)
        ub = np.array([LR_BOUNDS[1], max(NEURON_CHOICES), max(BATCH_CHOICES)], float)

        positions  = np.array([np.random.uniform(lb, ub) for _ in range(self.swarm_size)])
        velocities = np.zeros_like(positions)

        p_best_pos = positions.copy()
        p_best_fit = np.array([self._fitness(p) for p in positions])
        g_best_idx = np.argmin(p_best_fit)
        g_best_pos = p_best_pos[g_best_idx].copy()
        g_best_fit = p_best_fit[g_best_idx]

        self.convergence = [g_best_fit]

        for t in range(self.max_iter):
            r1, r2 = np.random.rand(2)
            for i in range(self.swarm_size):
                velocities[i] = (
                    self.omega * velocities[i]
                    + self.c1 * r1 * (p_best_pos[i] - positions[i])
                    + self.c2 * r2 * (g_best_pos    - positions[i])
                )
                positions[i] += velocities[i]
                positions[i], velocities[i], lb, ub = self._fix_bounds(
                    positions[i], velocities[i], lb.copy(), ub.copy())

                fit = self._fitness(positions[i])
                if fit < p_best_fit[i]:
                    p_best_fit[i] = fit
                    p_best_pos[i] = positions[i].copy()
                if fit < g_best_fit:
                    g_best_fit = fit
                    g_best_pos = positions[i].copy()

            self.convergence.append(g_best_fit)
            if verbose:
                lr, n, b = self._decode(g_best_pos)
                print(f'  Iter {t+1:02d} | best RMSE={g_best_fit:.4f} '
                      f'| lr={lr:.5f}, neurons={n}, batch={b}')

        self.best_lr, self.best_neurons, self.best_batch = self._decode(g_best_pos)
        self.best_fitness = g_best_fit
        return self.best_lr, self.best_neurons, self.best_batch


## 6. GA-Based Hyperparameter Optimisation

Each individual is an integer index triple `[lr_idx, neuron_idx, batch_idx]`  
selecting from discretised choice lists (see Section 3.3 of the paper).


In [ ]:
class GA:
    """
    Genetic Algorithm for LSTM hyperparameter search.

    Encoding  : integer index triple [lr_idx, neuron_idx, batch_idx]
    Crossover : single-point
    Mutation  : single-index replacement (rate = 0.3)
    Selection : roulette wheel
    Elitism   : top-2 individuals preserved each generation
    """

    LR_CHOICES = list(np.linspace(LR_BOUNDS[0], LR_BOUNDS[1], 10))

    def __init__(self, pop_size=POP_SIZE, max_gen=MAX_ITER, mut_rate=0.3):
        self.pop_size = pop_size
        self.max_gen  = max_gen
        self.mut_rate = mut_rate
        self.convergence = []

    # ── encoding helpers ──────────────────────────────────────────────────────
    def _random_individual(self):
        return [
            random.randint(0, len(self.LR_CHOICES)  - 1),
            random.randint(0, len(NEURON_CHOICES)   - 1),
            random.randint(0, len(BATCH_CHOICES)    - 1),
        ]

    def _decode(self, ind):
        return (self.LR_CHOICES[ind[0]],
                NEURON_CHOICES[ind[1]],
                BATCH_CHOICES[ind[2]])

    # ── fitness ───────────────────────────────────────────────────────────────
    def _fitness(self, ind):
        lr, neurons, batch = self._decode(ind)
        model = build_lstm(neurons, lr)
        train_model(model, X_train, y_train,
                    batch_size=batch, epochs=OPT_EPOCHS, patience=5)
        y_pred = model.predict(X_test, verbose=0).flatten()
        return np.sqrt(mean_squared_error(y_test, y_pred))

    # ── genetic operators ─────────────────────────────────────────────────────
    @staticmethod
    def _crossover(p1, p2):
        pt = random.randint(1, len(p1) - 1)
        return p1[:pt] + p2[pt:], p2[:pt] + p1[pt:]

    def _mutate(self, ind):
        if random.random() < self.mut_rate:
            i = random.randint(0, 2)
            choices = [len(self.LR_CHOICES), len(NEURON_CHOICES), len(BATCH_CHOICES)]
            ind[i]  = random.randint(0, choices[i] - 1)
        return ind

    @staticmethod
    def _roulette(population, fitnesses):
        inv = [1.0 / (f + 1e-9) for f in fitnesses]   # lower fitness = higher probability
        total = sum(inv)
        probs = [v / total for v in inv]
        return random.choices(population, weights=probs, k=2)

    # ── main loop ─────────────────────────────────────────────────────────────
    def run(self, verbose=False):
        population = [self._random_individual() for _ in range(self.pop_size)]
        fitnesses  = [self._fitness(ind) for ind in population]

        best_idx = np.argmin(fitnesses)
        best_fit = fitnesses[best_idx]
        best_ind = population[best_idx][:]
        self.convergence = [best_fit]

        for g in range(self.max_gen):
            # Elitism: carry top-2 unchanged
            sorted_pop = sorted(zip(fitnesses, population), key=lambda x: x[0])
            new_pop    = [ind[:] for _, ind in sorted_pop[:2]]

            while len(new_pop) < self.pop_size:
                p1, p2 = self._roulette(population, fitnesses)
                c1, c2 = self._crossover(p1[:], p2[:])
                new_pop.extend([self._mutate(c1), self._mutate(c2)])
            population = new_pop[:self.pop_size]
            fitnesses  = [self._fitness(ind) for ind in population]

            gen_best = min(fitnesses)
            if gen_best < best_fit:
                best_fit = gen_best
                best_ind = population[np.argmin(fitnesses)][:]

            self.convergence.append(best_fit)
            if verbose:
                lr, n, b = self._decode(best_ind)
                print(f'  Gen {g+1:02d} | best RMSE={best_fit:.4f} '
                      f'| lr={lr:.5f}, neurons={n}, batch={b}')

        self.best_lr, self.best_neurons, self.best_batch = self._decode(best_ind)
        self.best_fitness = best_fit
        return self.best_lr, self.best_neurons, self.best_batch


## 7. Run Optimisation (FD001)

> **Note:** Optimisation is performed once on FD001. The best configurations  
> are then used for multi-run evaluation across all sub-datasets (Section 8).


In [ ]:
# Load FD001 for optimisation
X_train, y_train, X_test, y_test = preprocess('FD001')
print(f'FD001 — Train: {X_train.shape}, Test: {X_test.shape}')

# ── PSO ─────────────────────────────────────────────────────────────────────
print('\n=== PSO Optimisation ===')
t0  = time.time()
pso = PSO()
pso_best_lr, pso_best_neurons, pso_best_batch = pso.run(verbose=True)
pso_time = (time.time() - t0) / 60
print(f'\nPSO done in {pso_time:.1f} min')
print(f'Best config: lr={pso_best_lr:.5f}, neurons={pso_best_neurons}, batch={pso_best_batch}')
print(f'Best validation RMSE: {pso.best_fitness:.4f}')

# ── GA ──────────────────────────────────────────────────────────────────────
print('\n=== GA Optimisation ===')
t0 = time.time()
ga = GA()
ga_best_lr, ga_best_neurons, ga_best_batch = ga.run(verbose=True)
ga_time = (time.time() - t0) / 60
print(f'\nGA done in {ga_time:.1f} min')
print(f'Best config: lr={ga_best_lr:.5f}, neurons={ga_best_neurons}, batch={ga_best_batch}')
print(f'Best validation RMSE: {ga.best_fitness:.4f}')


## 8. Multi-Run Statistical Validation

Each model configuration is evaluated over N independent runs with fixed but distinct seeds.  
N=10 for FD001/FD003; N=5 for FD002/FD004 (higher computational cost).


In [ ]:
results_rows = []

for dataset in DATASETS:
    print(f'\n===== {dataset} =====')
    X_train, y_train, X_test, y_test = preprocess(dataset)
    n_runs = N_RUNS if dataset in ['FD001', 'FD003'] else N_RUNS_MULTI

    model_configs = {
        'LSTM':     dict(neurons=32,              lr=0.001,          batch=32,              bidirectional=False),
        'GA-LSTM':  dict(neurons=ga_best_neurons,  lr=ga_best_lr,     batch=ga_best_batch,   bidirectional=False),
        'PSO-LSTM': dict(neurons=pso_best_neurons, lr=pso_best_lr,    batch=pso_best_batch,  bidirectional=False),
        'BiLSTM':   dict(neurons=32,              lr=0.001,          batch=32,              bidirectional=True),
    }

    for model_name, cfg in model_configs.items():
        print(f'  {model_name} ({n_runs} runs)...')
        for run in range(n_runs):
            tf.random.set_seed(run)
            np.random.seed(run)
            model = build_lstm(cfg['neurons'], cfg['lr'],
                               bidirectional=cfg['bidirectional'])
            train_model(model, X_train, y_train, batch_size=cfg['batch'])
            y_pred = model.predict(X_test, verbose=0).flatten()
            m = compute_metrics(y_test, y_pred)
            results_rows.append({'dataset': dataset, 'model': model_name,
                                 'run': run, **m})
            print(f'    run {run} — RMSE={m["RMSE"]:.4f}')

runs_df = pd.DataFrame(results_rows)
runs_df.to_csv(f'{OUT_DIR}/per_run_metrics.csv', index=False)
print('\nSaved: per_run_metrics.csv')


## 9. Summary Table (Mean ± SD, Best, Worst, CV)

In [ ]:
summary = (
    runs_df.groupby(['dataset', 'model'])
    .agg(Mean=('RMSE','mean'), SD=('RMSE','std'),
         Best=('RMSE','min'),  Worst=('RMSE','max'))
    .reset_index()
)
summary['CV (%)'] = (summary['SD'] / summary['Mean'] * 100).round(2)
summary = summary.round(4)
print(summary.to_string(index=False))
summary.to_csv(f'{OUT_DIR}/summary_table.csv', index=False)


## 10. Statistical Tests

Shapiro-Wilk normality test → if all groups normal and variances homogeneous (Levene):  
one-way ANOVA + Tukey HSD. Otherwise: Kruskal-Wallis + Dunn (Bonferroni).


In [ ]:
from scipy import stats
from itertools import combinations

try:
    from scikit_posthocs import posthoc_dunn
except ImportError:
    posthoc_dunn = None
    print('scikit-posthocs not found — install with: pip install scikit-posthocs')

models = runs_df['model'].unique()

for dataset in DATASETS:
    print(f'\n── {dataset} ──')
    sub = runs_df[runs_df['dataset'] == dataset]
    groups = [sub[sub['model'] == m]['RMSE'].values for m in models]

    # Shapiro-Wilk
    sw = {m: stats.shapiro(g) for m, g in zip(models, groups)}
    all_normal = all(r.pvalue > 0.05 for r in sw.values())
    print('  Shapiro-Wilk:', {m: f'p={r.pvalue:.3f}' for m, r in sw.items()})

    # Levene
    lev = stats.levene(*groups)
    print(f'  Levene: F={lev.statistic:.3f}, p={lev.pvalue:.3f}')

    if all_normal and lev.pvalue > 0.05:
        f, p = stats.f_oneway(*groups)
        print(f'  ANOVA: F={f:.3f}, p={p:.3f}')
    else:
        h, p = stats.kruskal(*groups)
        print(f'  Kruskal-Wallis: H={h:.3f}, p={p:.3f}')
        if posthoc_dunn is not None:
            dunn = posthoc_dunn(sub, val_col='RMSE', group_col='model', p_adjust='bonferroni')
            print('  Dunn (Bonferroni):')
            print(dunn.round(3))


## 11. Convergence Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(pso.convergence, color='steelblue', linewidth=2, label='PSO')
axes[0].set_title('PSO Convergence (FD001)', fontsize=13)
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('Best RMSE')
axes[0].legend()

axes[1].plot(ga.convergence, color='darkorange', linewidth=2, label='GA')
axes[1].set_title('GA Convergence (FD001)', fontsize=13)
axes[1].set_xlabel('Generation'); axes[1].set_ylabel('Best RMSE')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/convergence_FD001.pdf', dpi=150, bbox_inches='tight')
plt.show()


## 12. Box Plots (Multi-Run RMSE Distribution)

In [ ]:
model_order = ['LSTM', 'GA-LSTM', 'PSO-LSTM', 'BiLSTM']
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

for dataset in DATASETS:
    sub = runs_df[runs_df['dataset'] == dataset]
    data = [sub[sub['model'] == m]['RMSE'].values for m in model_order]

    fig, ax = plt.subplots(figsize=(8, 5))
    bp = ax.boxplot(data, patch_artist=True, labels=model_order)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(f'RMSE Distribution — {dataset}', fontsize=13)
    ax.set_ylabel('RMSE (cycles)')
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/boxplot_{dataset}.pdf', dpi=150, bbox_inches='tight')
    plt.show()


## 13. SHAP Explainability Analysis

SHAP attributions are computed over the **last timestep** of each input sequence.  
A `KernelExplainer` with 100 background samples is used (model-agnostic).  
See Section 4.4 of the paper for the full methodological description.


In [ ]:
# Retrain best-run models on FD001 for SHAP analysis
X_train, y_train, X_test, y_test = preprocess('FD001')

best_models = {}
configs_shap = {
    'LSTM':     dict(neurons=32,              lr=0.001,        batch=32,             bidirectional=False),
    'GA-LSTM':  dict(neurons=ga_best_neurons,  lr=ga_best_lr,   batch=ga_best_batch,  bidirectional=False),
    'PSO-LSTM': dict(neurons=pso_best_neurons, lr=pso_best_lr,  batch=pso_best_batch, bidirectional=False),
    'BiLSTM':   dict(neurons=32,              lr=0.001,        batch=32,             bidirectional=True),
}
for name, cfg in configs_shap.items():
    tf.random.set_seed(0); np.random.seed(0)
    m = build_lstm(cfg['neurons'], cfg['lr'], bidirectional=cfg['bidirectional'])
    train_model(m, X_train, y_train, batch_size=cfg['batch'])
    best_models[name] = m
    print(f'{name} trained.')

# ── SHAP wrapper ─────────────────────────────────────────────────────────────
# Last timestep: X[:, -1, :] → shape (n_samples, n_features)
X_2d  = X_test[:, -1, :]
X_bg  = X_train[:100, -1, :]

def make_predict_fn(model, window=WINDOW):
    """Wrap model.predict so it accepts 2D input (last-timestep)."""
    def fn(x):
        x3d = np.tile(x[:, np.newaxis, :], (1, window, 1))
        return model.predict(x3d, verbose=0).flatten()
    return fn

shap_vals = {}
for name, model in best_models.items():
    print(f'Computing SHAP for {name}...')
    explainer = shap.KernelExplainer(make_predict_fn(model), X_bg)
    shap_vals[name] = explainer.shap_values(X_2d[:200], nsamples=100)
    print(f'  Done. Shape: {shap_vals[name].shape}')

print('All SHAP values computed.')


In [ ]:
# Feature names (FD001: 14 features, no op settings)
feature_names = [c for c in
    ['T2','T24','T30','T50','P2','P15','P30','Nf','Nc','Ps30','phi',
     'NRf','NRc','BPR','farB','htBleed','Nf_dmd','PCNfR_dmd','W31','W32']
    if c not in ['s1','s5','s6','s10','s16','s17','s18','s19']]
feature_names = feature_names[:X_2d.shape[1]]   # trim to actual feature count

# ── Mean absolute SHAP bar chart ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 6), sharey=True)
colors_shap = {'LSTM':'#4C72B0','GA-LSTM':'#DD8452',
               'PSO-LSTM':'#55A868','BiLSTM':'#C44E52'}
for ax, (name, sv) in zip(axes, shap_vals.items()):
    mean_abs = np.abs(sv).mean(axis=0)
    top10_idx = np.argsort(mean_abs)[-10:][::-1]
    ax.barh([feature_names[i] for i in top10_idx],
            mean_abs[top10_idx], color=colors_shap[name])
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('Mean |SHAP|')
plt.suptitle('Top-10 Feature Importance — FD001', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/shap_bar_FD001.pdf', dpi=150, bbox_inches='tight')
plt.show()


## 14. Output Files

In [ ]:
print('Generated outputs:')
for f in sorted(os.listdir(OUT_DIR)):
    path = os.path.join(OUT_DIR, f)
    print(f'  {f:40s} {os.path.getsize(path):>10,} bytes')
